# Data Management - Oxford 102 flowers

### Import libraries

In [ ]:
import os
from scipy import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from download_dataset import download_dataset

In [2]:
download_dataset()

Oxford 102 Flowers dataset is ready.


### Define custom OxfordFlowersDataset

In [ ]:
class OxfordFlowersDataset(Dataset):
  # Setup: where to find images and labels
  def __init__(self, root_dir, transform=None):
    """
      Args:
          root_dir (str): Path to oxford_102_flowers directory
          transform: torchvision transforms to apply to images
    """
    self.root_dir = root_dir
    self.transform = transform

    # Load labels from .mat file
    labels_path = os.path.join(root_dir, "imagelabels.mat")
    labels_mat = io.loadmat(labels_path)
    self.labels = labels_mat['labels'][0]
    
    # image directory
    self.images_dir = os.path.join(root_dir, "jpg")

    # number of samples
    self.num_samples = len(self.labels)

  # How many total samples are in the dataset
  def __len__(self):
    return self.num_samples

  # How to get image and label number 'idx'
  def __getitem__(self, idx):
    # Image files are 1-indexed: image_00001.jpg
    img_name = f"image_{idx + 1:05d}.jpg"
    img_path = os.path.join(self.images_dir, img_name)

    image = Image.open(img_path).convert("RGB")
    label = int(self.labels[idx]) - 1  # Convert to 0-indexed

    if self.transform:
      image = self.transform(image)

    return image, label

### Test the OxfordFlowersDataset class

In [5]:
dataset = OxfordFlowersDataset(root_dir="oxford_102_flowers")
print(f"Dataset size: {len(dataset)} samples")
img, label = dataset[0]
print(f"First image size: {img.size}, Label: {label}")

Dataset size: 8189 samples
First image size: (591, 500), Label: 76


### Check for image sizes
- Images are of different sizes and pyTorch expects images to be in same size
- 

In [6]:
for i in [1, 50, 100, 250, 500]:
    img, label = dataset[i]
    print(f"Image {i} size: {img.size}, Label: {label}")
    print(f"Type: {type(img)}")

Image 1 size: (625, 500), Label: 76
Type: <class 'PIL.Image.Image'>
Image 50 size: (666, 500), Label: 76
Type: <class 'PIL.Image.Image'>
Image 100 size: (588, 500), Label: 76
Type: <class 'PIL.Image.Image'>
Image 250 size: (667, 500), Label: 76
Type: <class 'PIL.Image.Image'>
Image 500 size: (667, 500), Label: 87
Type: <class 'PIL.Image.Image'>


### Changing image size (transform)

In [7]:
img, _ = dataset[0]
print(f"Original image size: {img.size}")
transform_flowers = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
      mean=[0.485, 0.456, 0.406],
      std=[0.229, 0.224, 0.225])
])

img = transform_flowers(img)
print(img.shape) # Should print torch.Size([3, 224, 224])
print(img[0, :3, :3]) # Print a small patch of the first channel


Original image size: (591, 500)
torch.Size([3, 224, 224])
tensor([[-0.4568, -0.4739, -0.4739],
        [-0.4397, -0.4739, -0.4739],
        [-0.3883, -0.4226, -0.4568]])


In [8]:
dataset = OxfordFlowersDataset(root_dir="oxford_102_flowers", transform=transform_flowers)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True)  

for images, labels in dataloader:
    print(f"Batch image tensor size: {images.size()}")  # Should be [4, 3, 224, 224]
    print(f"Batch labels tensor size: {labels.size()}")  # Should be [4]
    break  # Just process one batch for demonstration


Batch image tensor size: torch.Size([4, 3, 224, 224])
Batch labels tensor size: torch.Size([4])


### Quick sanity check

In [9]:
img, label = dataset[0]
print(f"Shape: {img.shape}, Label: {label}")
print(f"Type: {img.dtype}, Type of label: {type(label)}")
print(f"Range: {img.min():.1f} to {img.max():.1f}")

Shape: torch.Size([3, 224, 224]), Label: 76
Type: torch.float32, Type of label: <class 'int'>
Range: -2.1 to 2.6


### Splitting dataset in training, validation and test sets

In [10]:
train_size = int(0.7 * len(dataset))
validation_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - validation_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, validation_size, test_size]
)

print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

Train set size: 5732
Validation set size: 1228
Test set size: 1229


### Batching - DataLoader

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

for images, labels in train_loader:
    print(f"Train batch image tensor size: {images.size()}")  # Should be [32, 3, 224, 224]
    print(f"Train batch labels tensor size: {labels.size()}")  # Should be [32]
    break  # Just process one batch for demonstration


Train batch image tensor size: torch.Size([32, 3, 224, 224])
Train batch labels tensor size: torch.Size([32])
